Make sure to upload requirements.txt from the project repository, to create the 'dataset' folder and place your csv training data there.

## Setup and Imports

Install the necessary libraries.

In [1]:
!pip install -r requirements.txt

## Feature Extraction and Data Augmentation

Extract features from raw IMU data using a sliding window. Each physical window is used to generate augmented, synthetical data that massively increases dataset sizes. The resulting `features.csv` file contains the extracted features and the label of each sample, the latter being generated directly from the original csv filename in order to be modular.

Since our arms were hurting after throwing so many punches to gather the training data, we generated synthetic data starting from our (smaller amount of) collected samples. These new samples are obtained from randomly scaling, stretching and adding noise to the collected samples.

The features are then extracted from each row of data (real + augmented) and for each axis, and they are:

- mean 
- standard deviation
- root mean square
- minimum
- maximum
- power spectral density peak

These amount to 36 features (6 features for 6 axes), which reach 38 when adding:

- X and Y axes correlation for the gyroscope
- X and Y axes correlation for the accelerometer

which help in detecting gestures like circles.


In [25]:
import pandas as pd
import numpy as np
from scipy.signal import welch
from scipy.interpolate import interp1d
import os
import glob

WINDOW_SIZE = 60 # 60 samples window size (~0.5 seconds considering the IMU rate of 119hz)
STEP_SIZE = 30   # 50% overlap for the windows

DATA_DIR = 'dataset'
OUTPUT_FILE = 'features.csv' 

NUM_AUGMENTATIONS = 3  # Number of iterations of augmentation per window
NOISE_LEVEL = 0.05     
SCALE_RANGE = (0.8, 1.2)
TIME_STRETCH_RANGE = (0.8, 1.2)

global_window_id = 0 # global counter for windows that allows us to split data and their augmented data in groups
                     # and avoid using in the test set data related to the one in the training set

def add_jitter(data, noise_level=0.05):
    """ Adds noise to the data"""
    return data + np.random.normal(0, noise_level, data.shape)

def scale_data(data, scale_range=(0.8, 1.2)):
    """ Scales the data to increase or decrease gesture amplitude"""
    return data * np.random.uniform(scale_range[0], scale_range[1])

def time_stretch(data, stretch_range=(0.8, 1.2)):
    """ Makes the gesture faster or slower """
    factor = np.random.uniform(stretch_range[0], stretch_range[1])
    orig_steps = np.arange(len(data))
    new_length = int(len(data) * factor)

    if new_length < 16: return data # Prevent errors on tiny windows

    new_steps = np.linspace(0, len(data) - 1, new_length)
    stretched_data = np.zeros((new_length, data.shape[1]))
    for i in range(data.shape[1]):
        stretched_data[:, i] = interp1d(orig_steps, data[:, i], kind='linear')(new_steps)
    return stretched_data

def generate_augmentations(data, num_augments):
    """ Applies data augmentation to the data window"""
    data_versions = [data]
    for _ in range(num_augments):
        aug_data = data.copy()
        if np.random.rand() > 0.5: aug_data = add_jitter(aug_data, NOISE_LEVEL)
        if np.random.rand() > 0.5: aug_data = scale_data(aug_data, SCALE_RANGE)
        if np.random.rand() > 0.5: aug_data = time_stretch(aug_data, TIME_STRETCH_RANGE)
        data_versions.append(aug_data)
    return data_versions

def compute_features(window):
    """ Computes features for the raw data window"""
    mean_val, std_val = np.mean(window), np.std(window)
    rms_val = np.sqrt(np.mean(window**2))
    min_val, max_val = np.min(window), np.max(window)
    
    if np.all(window == window[0]): 
        psd_peak = 0.0
    else:
        # Cap length at 64 to prevent overflows from time_stretch augmentation
        safe_len = min(len(window), 64)
        
        # Match Arduino FFT PSD extraction exactly
        padded = np.zeros(64)
        padded[:safe_len] = window[:safe_len]
        
        # Arduino FFT Hamming window formula
        hamming = 0.54 - 0.46 * np.cos(2.0 * np.pi * np.arange(64) / 63)
        padded *= hamming
        
        # Compute FFT and match Arduino's power magnitude formula
        fft_mags = np.abs(np.fft.fft(padded))
        power = (fft_mags[1:32] ** 2) / 60.0
        psd_peak = np.max(power) if len(power) > 0 else 0.0
        
    return [mean_val, std_val, rms_val, min_val, max_val, psd_peak]

def process_file(filepath, label):
    """ Process an entire CSV file of raw data"""
    global global_window_id
    if not os.path.exists(filepath): return []
    df = pd.read_csv(filepath)
    cols = ['aX', 'aY', 'aZ', 'gX', 'gY', 'gZ']
    original_data = df[cols].values
    features_list = []
    
    # Slide window over the raw data
    for i in range(0, len(original_data) - WINDOW_SIZE + 1, STEP_SIZE):
        base_window = original_data[i:i + WINDOW_SIZE]
        
        # Generate augmentations for this window
        window_versions = generate_augmentations(base_window, NUM_AUGMENTATIONS)
        # window_versions = generate_augmentations(base_window, NUM_AUGMENTATIONS)
        
        for window in window_versions:
            window_features = []
            for axis_idx in range(6): 
                window_features.extend(compute_features(window[:, axis_idx]))
            
            with np.errstate(divide='ignore', invalid='ignore'):
                corr_aXY = np.corrcoef(window[:, 0], window[:, 1])[0, 1]
                corr_gXY = np.corrcoef(window[:, 3], window[:, 4])[0, 1]
                window_features.extend([0.0 if np.isnan(corr_aXY) else corr_aXY, 
                                        0.0 if np.isnan(corr_gXY) else corr_gXY])
            
            window_features.append(global_window_id) # The unique ID for this window and its twins
            window_features.append(label)
            features_list.append(window_features)
            
        global_window_id += 1 # Increment for the next window
        
    return features_list

all_features = []
if not os.path.exists(DATA_DIR):
    print(f"Please create the '{DATA_DIR}' directory and add your CSV files there.")
else:
    for filepath in glob.glob(f'{DATA_DIR}/*.csv'):
        label = os.path.basename(filepath).split('.')[0].split('_')[0]
        print(f"Processing {filepath}...")
        all_features.extend(process_file(filepath, label))

    if all_features:
        axes, feature_names = ['aX','aY','aZ','gX','gY','gZ'], ['mean','std','rms','min','max','psd']
        col_names = [f"{axis}_{fname}" for axis in axes for fname in feature_names] + ['corr_aXY', 'corr_gXY', 'window_id', 'label']
        pd.DataFrame(all_features, columns=col_names).to_csv(OUTPUT_FILE, index=False)
        print(f"Feature extraction complete. Extracted {len(all_features)} windows (original + augmented).")
        print(f"Total unique physical window segments: {global_window_id}")
    else:
        print(f"No features were extracted. Please check you placed your data in {DATA_DIR}.")

Processing dataset/circle.csv...
Processing dataset/left-right.csv...
Processing dataset/punch.csv...
Processing dataset/up-down.csv...
Feature extraction complete. Extracted 6320 windows (original + augmented).
Total unique physical window segments: 1580


## Neural Network Training

Train a minimal neural network on the collected samples, after splitting the set based on the window_id/group, normalizing data and using dropout and early stopping mechanics to avoid overfitting. The size of the output layer itself is based on the number of extracted features, so once again you do not need to edit anything to add new features.

The model will be exported both as a `.tflite` file and as a C++/Arduino compatible header file (`.h`). Make sure to transfer the latter to the inference folder before attempting to load the Arduino sketch to the board.

In [22]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix
import textwrap
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Suppresses INFO, WARNING, and ERROR logs

tf.get_logger().setLevel('ERROR')        # Suppresses Python-level warnings

INPUT_FILE = 'features.csv'
HEADER_FILE = 'gesture_recognition.h'

print("Loading data...")
df = pd.read_csv(INPUT_FILE)

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
gesture_names = label_encoder.classes_

# Split into Features (X), Labels (Y), and Groups (window_id)
# We drop 'label' and 'window_id' so the model only sees sensor data
X = df.drop(['label', 'window_id'], axis=1, errors='ignore').values
Y = df['label'].values
groups = df['window_id'].values

num_classes = len(np.unique(Y))

# Use GroupShuffleSplit to keep all augmentations of a specific window in the same set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, Y, groups=groups)) # <- notice it allows us to specify groups

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = Y[train_idx], Y[test_idx]

print(f"Total feature rows: {len(df)}")
print(f"Using: {len(np.unique(groups[train_idx]))} windows for training,")
print(f"       {len(np.unique(groups[test_idx]))} windows for testing.")

# Normalize the trainig set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nTraining Neural Network...")
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Early Stopping to terminate training automatically (100 epochs is usually way too many)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_scaled, y_train, epochs=100, batch_size=16, validation_split=0.2, callbacks=[early_stop], verbose=0)

print("\nEvaluating Model...")
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Export to TFLite and C++
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

hex_array_str = ', '.join([format(val, '#04x') for val in tflite_model])
wrapped_hex_array = textwrap.fill(hex_array_str, width=100)
scaler_means_str = ', '.join([str(val) for val in scaler.mean_])
scaler_scales_str = ', '.join([str(val) for val in scaler.scale_])
labels_str = ', '.join([f'"{name}"' for name in gesture_names])

c_code = f"""// Automatically generated model and scaler data
#ifndef MODEL_DATA_H
#define MODEL_DATA_H

const int NUM_CLASSES = {len(gesture_names)};
const char* const GESTURE_LABELS[] = {{{labels_str}}};
const int NUM_FEATURES = {X_train.shape[1]};

const float scaler_mean[{X_train.shape[1]}] = {{{scaler_means_str}}};
const float scaler_scale[{X_train.shape[1]}] = {{{scaler_scales_str}}};

const unsigned int g_model_len = {len(tflite_model)};
alignas(8) const unsigned char g_model[] = {{
{wrapped_hex_array}
}};

#endif // MODEL_DATA_H\n"""

with open(HEADER_FILE, 'w') as f:
    f.write(c_code)
with open(HEADER_FILE.removesuffix(".h")+".tflite", "wb") as f:
    f.write(tflite_model)

print(f"\nSuccess! C++ Header exported to {HEADER_FILE}")

Loading data...
Total feature rows: 6320
Using: 1264 windows for training,
       316 windows for testing.

Training Neural Network...

Evaluating Model...
Test Accuracy: 95.65%
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Confusion Matrix:
[[329   3   4   0]
 [  0 336   8  12]
 [  0   8 304   0]
 [  0   4  16 240]]
Saved artifact at '/tmp/tmpr9zvldo8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 38), dtype=tf.float32, name='keras_tensor_39')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  140535189629136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140535189625680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140535179441552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140535179428112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140535179442896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140535179442128: TensorSpec(shape=(), dtype=tf.resource, 

W0000 00:00:1773776088.935516  151083 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773776088.935550  151083 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773776088.935817  151083 reader.cc:83] Reading SavedModel from: /tmp/tmpr9zvldo8
I0000 00:00:1773776088.936831  151083 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773776088.936846  151083 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpr9zvldo8
I0000 00:00:1773776088.942893  151083 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773776088.979438  151083 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpr9zvldo8
I0000 00:00:1773776088.990698  151083 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 54901 microseconds.


# Testing the model 
If you quickly want to check that the model doesn't spout out random data (or worse, straight up crashes), you can invoke it here, before flashing it to the device.

In [20]:
import tensorflow as tf
import numpy as np
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Change this to the path of your actual .tflite file
model_path = HEADER_FILE.removesuffix(".h")+".tflite"

interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Feed it 38 clean dummy floats
input_shape = input_details[0]['shape']
sample_data = np.array([[
    1.3354, -0.6313, -0.6647, 0.8438, 0.2004, -0.5546, 0.4229, -0.2298, -1.1134, -0.3311, 
    2.4345, -0.7655, 1.6291, -0.1292, 1.4522, 1.4805, 1.6758, -0.1190, 0.2969, 0.5650, 
    0.2942, -0.7829, 1.9567, -0.5337, 0.7300, -0.4129, -0.3910, 0.9033, 0.6414, -0.6748, 
    0.4381, -0.5680, -0.5857, 1.0221, 0.5087, -0.5542, 0.7537, 0.4789
]], dtype=np.float32)

interpreter.set_tensor(input_details[0]['index'], sample_data)
interpreter.invoke()

output_data = interpreter.get_tensor(output_details[0]['index'])
print("\nModel Output:", output_data)


Model Output: [[8.4219238e-07 6.0592527e-08 9.9964976e-01 3.4937199e-04]]
